# Double Q-Learning Exploration Strategies for Chemotherapy Scheduling
## Reproducible Google Colab implementation

**Research question:** With the same reconstructed PK/PD simulator, state discretisation, reward, action space and Double Q-learning update, how do (1) ε-greedy, (2) sigmoid-decayed ε-greedy, and (3) Upper Confidence Bound (UCB) exploration compare?

**Important scope:** This is a computational reproduction/extension of the supplied paper, *Double Q-Learning for Intelligent Multi-Drug Scheduling in Cancer Chemotherapy Optimisation* (Alizade & Hajipour, 2026). The paper's stated model is a simulated three-drug PK/PD system with eight resistance compartments, a 84-day training horizon in the reported setup, and toxicity-aware rewards. The exploration-policy comparison below is a proposed methodological extension, not a result claimed by the paper. This is not a clinical tool or dosing recommendation.

### Notebook roadmap
1. Define model parameters and 343 discrete actions.
2. Implement the eight-compartment tumour model and PK/toxicity dynamics.
3. Define the RL environment, state bins, and reward.
4. Implement Double Q-learning with interchangeable exploration strategies.
5. Run a small smoke test, then a fair multi-seed comparison.
6. Plot reward, terminal tumour burden, toxicity, and exploration behaviour.
7. Save results to CSV for your presentation.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product
from collections import defaultdict
import time, math, random, os

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

## 1. Parameters and action space

The paper lists dose levels `{0, 5, 10, 20, 30, 40, 50}` for each of three drugs, yielding \(7^3=343\) joint actions. Parameters below are taken from the supplied paper's parameter table (Table 2) where available. Initial resistant compartments are zero in this reconstruction, matching the initialisation in the supplied starter notebook.

**Reproduction note:** The paper's full simulator and exact implementation are not provided as code. This notebook is a transparent reconstruction from its equations, not a bit-for-bit reproduction. The integration and reward implementation are explicitly documented below.

In [ ]:
DRUG_NAMES = ["Cisplatin", "Docetaxel", "Irinotecan"]
DOSE_LEVELS = [0, 5, 10, 20, 30, 40, 50]
ACTIONS = np.array(list(product(DOSE_LEVELS, repeat=3)), dtype=float)
N_ACTIONS = len(ACTIONS)
INITIAL_TUMOUR = 4.60517e11

# PK/PD parameters reported in the paper
RHO = 0.0099
PHI = np.array([0.008, 0.010, 0.014], dtype=float)
KILL = np.array([0.0084, 0.0074, 0.0092], dtype=float)
C_THRESHOLD = np.array([10.0, 10.0, 10.0], dtype=float)
LAMBDA = np.array([0.32, 0.27, 0.25], dtype=float)
BETA = np.array([0.40, 0.50, 0.45], dtype=float)

ALPHA = 0.01
GAMMA = 0.99
MAX_DAYS = 84
TOXICITY_LIMIT = 100.0  # per-drug threshold from the paper; reward penalises total toxicity above this value in this reconstruction
W1, W2 = 50.0, 0.5

print("Joint actions:", N_ACTIONS)
print("First actions:", ACTIONS[:5])
assert N_ACTIONS == 343

## 2. PK/PD simulator

The eight tumour compartments are \(S,R_1,R_2,R_3,R_{12},R_{13},R_{23},R_{123}\). Drug concentration follows first-order elimination, and cumulative toxicity follows concentration input minus clearance. We use explicit substeps to reduce numerical integration error.

The implementation is intended for algorithm comparison. The exact reward and toxicity aggregation in the paper are not completely unambiguous in the extracted text, so this notebook makes its choice explicit: tumour reward is proportional reduction in tumour burden; toxicity penalty applies when the **sum** of the three toxicity states exceeds 100. Treat that as a modelling assumption and test alternatives in sensitivity analysis.

In [ ]:
def tumour_derivatives(cells, concentrations):
    S, R1, R2, R3, R12, R13, R23, R123 = cells
    E = np.maximum(np.asarray(concentrations) - C_THRESHOLD, 0.0)
    E1, E2, E3 = E
    p1, p2, p3 = PHI
    k1, k2, k3 = KILL
    return np.array([
        RHO*(1-p1-p2-p3)*S - k1*E1*S - k2*E2*S - k3*E3*S,
        RHO*((1-p2-p3)*R1 + p1*S) - (k2*E2+k3*E3)*R1,
        RHO*((1-p1-p3)*R2 + p2*S) - (k1*E1+k3*E3)*R2,
        RHO*((1-p1-p2)*R3 + p3*S) - (k1*E1+k2*E2)*R3,
        RHO*((1-p3)*R12 + p2*R1 + p1*R2) - k3*E3*R12,
        RHO*((1-p2)*R13 + p3*R1 + p1*R3) - k2*E2*R13,
        RHO*((1-p1)*R23 + p2*R3 + p3*R2) - k1*E1*R23,
        RHO*R123 + RHO*(p1*R23 + p2*R13 + p3*R12)
    ], dtype=float)

def simulate_one_day(cells, concentrations, toxicities, action, substeps=100):
    cells = np.asarray(cells, dtype=float).copy()
    concentrations = np.asarray(concentrations, dtype=float).copy()
    toxicities = np.asarray(toxicities, dtype=float).copy()
    action = np.asarray(action, dtype=float)
    h = 1.0 / substeps
    for _ in range(substeps):
        cells += h * tumour_derivatives(cells, concentrations)
        concentrations += h * (action - LAMBDA * concentrations)
        toxicities += h * (concentrations - BETA * toxicities)
        if not (np.all(np.isfinite(cells)) and np.all(np.isfinite(concentrations))
                and np.all(np.isfinite(toxicities))):
            raise FloatingPointError("Non-finite value in simulator.")
        # Small negative numerical values are clipped only at machine-scale/rounding level.
        if np.min(cells) < -1e-5 or np.min(concentrations) < -1e-8:
            raise FloatingPointError("Negative state detected; inspect integration stability.")
        cells = np.maximum(cells, 0.0)
        concentrations = np.maximum(concentrations, 0.0)
        toxicities = np.maximum(toxicities, 0.0)
    return cells, concentrations, toxicities

# Basic numerical sanity check
test_cells = np.zeros(8); test_cells[0] = INITIAL_TUMOUR
test_out = simulate_one_day(test_cells, np.zeros(3), np.zeros(3), np.array([0.,0.,0.]))
print("One-day no-dose tumour:", f"{test_out[0].sum():.4e}")
print("All outputs finite:", all(np.all(np.isfinite(x)) for x in test_out))

## 3. Environment, reward, and state discretisation

The paper's state is total tumour burden plus three toxicity states. Since tabular Q-learning needs discrete keys, tumour burden is normalised by the initial tumour and each variable is binned. This discretisation is a design choice, and its resolution can materially affect results.

The 343-action space is large for tabular learning; many state-action pairs may remain unseen. This is important when interpreting a short training run.

In [ ]:
TUMOUR_BINS = np.array([0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 1.00, 1.25, 1.50])
TOXICITY_BINS = np.array([10, 25, 50, 100, 200, 400])

def discretize_state(state):
    tumour, t1, t2, t3 = state
    ratio = max(float(tumour), 0.0) / INITIAL_TUMOUR
    return (int(np.digitize(ratio, TUMOUR_BINS)),
            int(np.digitize(t1, TOXICITY_BINS)),
            int(np.digitize(t2, TOXICITY_BINS)),
            int(np.digitize(t3, TOXICITY_BINS)))

class ChemotherapyEnv:
    def __init__(self, max_days=MAX_DAYS, toxicity_limit=TOXICITY_LIMIT, substeps=100):
        self.max_days = max_days
        self.toxicity_limit = toxicity_limit
        self.substeps = substeps
        self.reset()

    def reset(self):
        self.cells = np.zeros(8, dtype=float)
        self.cells[0] = INITIAL_TUMOUR
        self.concentrations = np.zeros(3, dtype=float)
        self.toxicities = np.zeros(3, dtype=float)
        self.day = 0
        return self.get_state()

    def get_state(self):
        return np.array([self.cells.sum(), *self.toxicities], dtype=float)

    def step(self, action_id):
        if self.day >= self.max_days:
            raise RuntimeError("Episode ended; call reset().")
        action = ACTIONS[int(action_id)]
        prev_tumour = max(self.cells.sum(), 1e-12)
        self.cells, self.concentrations, self.toxicities = simulate_one_day(
            self.cells, self.concentrations, self.toxicities, action, self.substeps
        )
        curr_tumour = self.cells.sum()
        total_toxicity = self.toxicities.sum()
        tumour_reward = W1 * (prev_tumour - curr_tumour) / prev_tumour
        toxicity_penalty = W2 * max(0.0, total_toxicity - self.toxicity_limit)
        reward = float(tumour_reward - toxicity_penalty)
        self.day += 1
        done = self.day >= self.max_days
        return self.get_state(), reward, done

env = ChemotherapyEnv(max_days=3, substeps=20)
s = env.reset()
s2, r, d = env.step(0)
print("State key:", discretize_state(s), "Reward:", round(r, 4), "Done:", d)

## 4. Exploration strategies

We compare three strategies while keeping the Double Q-learning update unchanged:

1. **ε-greedy baseline:** exponentially decays ε from 1.0 to 0.01.
2. **Sigmoid ε schedule:** ε decreases smoothly around the middle of training, rather than decaying exponentially from the first episode.
3. **UCB action selection:** selects the action with the highest estimated value plus an uncertainty bonus \(c\sqrt{\log(t+1)/(N(s,a)+\epsilon)}\). UCB is applied to the combined Double-Q estimate. In a state not yet visited, all action counts are zero and the bonus encourages exploration.

Important: “sigmoid” here refers to a **sigmoid-shaped exploration schedule**, not a sigmoid transformation of Q-values. UCB is a distinct action-selection rule and does not use ε.

In [ ]:
def make_q_table():
    return defaultdict(lambda: np.zeros(N_ACTIONS, dtype=np.float64))

def make_count_table():
    return defaultdict(lambda: np.zeros(N_ACTIONS, dtype=np.int64))

def greedy_action(state, QA, QB, rng):
    q = QA[state] + QB[state]
    candidates = np.flatnonzero(np.isclose(q, q.max()))
    return int(rng.choice(candidates))

def select_action(strategy, state, QA, QB, counts, epsilon, total_steps, rng,
                  ucb_c=1.5):
    q = QA[state] + QB[state]
    if strategy in ("epsilon_greedy", "sigmoid"):
        if rng.random() < epsilon:
            return int(rng.integers(N_ACTIONS)), "explore"
        return greedy_action(state, QA, QB, rng), "exploit"
    elif strategy == "ucb":
        n = counts[state]
        bonus = ucb_c * np.sqrt(np.log(total_steps + 2.0) / (n + 1e-6))
        scores = q + bonus
        candidates = np.flatnonzero(np.isclose(scores, scores.max()))
        return int(rng.choice(candidates)), "ucb"
    raise ValueError(f"Unknown strategy: {strategy}")

def epsilon_for_episode(strategy, episode, n_episodes,
                        eps_start=1.0, eps_min=0.01, exp_decay=0.995):
    if strategy == "epsilon_greedy":
        return max(eps_min, eps_start * (exp_decay ** episode))
    if strategy == "sigmoid":
        # Midpoint at 55% of training; slope controls transition sharpness.
        midpoint = 0.55 * max(n_episodes - 1, 1)
        slope = 10.0 / max(n_episodes - 1, 1)
        z = np.clip(slope * (episode - midpoint), -60, 60)
        return eps_min + (eps_start - eps_min) / (1.0 + np.exp(z))
    if strategy == "ucb":
        return 0.0
    raise ValueError(strategy)

# Plot the schedules before training
n_demo = 200
for strategy in ["epsilon_greedy", "sigmoid", "ucb"]:
    vals = [epsilon_for_episode(strategy, ep, n_demo) for ep in range(n_demo)]
    plt.plot(vals, label=strategy)
plt.xlabel("Episode"); plt.ylabel("ε (not used by UCB)")
plt.title("Exploration schedules"); plt.legend(); plt.grid(alpha=.25); plt.show()

## 5. Double Q-learning update

The two Q-tables reduce maximisation bias by using one table to choose the next action and the other to evaluate it. The update rule is identical for all exploration methods; only action selection changes.

In [ ]:
def double_q_update(state, action_id, reward, next_state, done, QA, QB, alpha, gamma, rng):
    if rng.random() < 0.5:
        best_next = int(np.argmax(QA[next_state]))
        target = reward if done else reward + gamma * QB[next_state][best_next]
        QA[state][action_id] += alpha * (target - QA[state][action_id])
    else:
        best_next = int(np.argmax(QB[next_state]))
        target = reward if done else reward + gamma * QA[next_state][best_next]
        QB[state][action_id] += alpha * (target - QB[state][action_id])

## 6. Training and evaluation functions

For fair comparison, each strategy gets fresh Q-tables and its own reproducible random generator. The primary comparison uses the same episode count, horizon, learning rate, discount factor, environment, and set of random seeds. Training is intentionally configurable: start with a small smoke test, then increase episodes if runtime permits.

In [ ]:
def train_agent(strategy, n_episodes=100, seed=42, max_days=MAX_DAYS,
                alpha=ALPHA, gamma=GAMMA, epsilon_start=1.0,
                epsilon_min=0.01, exp_decay=0.995, ucb_c=1.5,
                substeps=20, verbose=False):
    rng = np.random.default_rng(seed)
    QA, QB, counts = make_q_table(), make_q_table(), make_count_table()
    env = ChemotherapyEnv(max_days=max_days, substeps=substeps)
    episode_rewards, final_tumours, final_toxicities = [], [], []
    epsilon_history, exploration_fraction = [], []
    total_steps = 0

    for ep in range(n_episodes):
        state = discretize_state(env.reset())
        epsilon = epsilon_for_episode(strategy, ep, n_episodes,
                                      epsilon_start, epsilon_min, exp_decay)
        episode_rewards.append(0.0)
        exploratory = 0

        for _ in range(max_days):
            action_id, decision = select_action(
                strategy, state, QA, QB, counts, epsilon, total_steps, rng, ucb_c
            )
            if decision == "explore":
                exploratory += 1
            next_continuous, reward, done = env.step(action_id)
            next_state = discretize_state(next_continuous)
            double_q_update(state, action_id, reward, next_state, done,
                            QA, QB, alpha, gamma, rng)
            counts[state][action_id] += 1
            episode_rewards[-1] += reward
            state = next_state
            total_steps += 1
            if done:
                break

        final_tumours.append(float(env.cells.sum()))
        final_toxicities.append(float(env.toxicities.sum()))
        epsilon_history.append(epsilon)
        exploration_fraction.append(exploratory / max_days)
        if verbose and (ep + 1) % max(1, n_episodes // 10) == 0:
            print(f"{strategy}: episode {ep+1}/{n_episodes}, "
                  f"reward={episode_rewards[-1]:.3f}, epsilon={epsilon:.3f}")

    return {
        "strategy": strategy, "seed": seed, "QA": QA, "QB": QB, "counts": counts,
        "rewards": np.array(episode_rewards), "tumours": np.array(final_tumours),
        "toxicities": np.array(final_toxicities), "epsilons": np.array(epsilon_history),
        "exploration_fraction": np.array(exploration_fraction),
        "unique_states": len(set(QA.keys()) | set(QB.keys())),
        "visited_state_action_pairs": int(sum(np.count_nonzero(v) for v in counts.values()))
    }

def evaluate_greedy_policy(agent, max_days=MAX_DAYS, seed=123, substeps=20):
    # Evaluation is greedy for all methods: no random exploration bonus.
    rng = np.random.default_rng(seed)
    env = ChemotherapyEnv(max_days=max_days, substeps=substeps)
    state = discretize_state(env.reset())
    rows = []
    for day in range(max_days):
        action_id = greedy_action(state, agent["QA"], agent["QB"], rng)
        next_state_cont, reward, done = env.step(action_id)
        rows.append({
            "Day": day + 1,
            "Action_ID": action_id,
            "Dose_Cisplatin": ACTIONS[action_id, 0],
            "Dose_Docetaxel": ACTIONS[action_id, 1],
            "Dose_Irinotecan": ACTIONS[action_id, 2],
            "Tumour_Cells": env.cells.sum(),
            "Tumour_Ratio": env.cells.sum() / INITIAL_TUMOUR,
            "Toxicity_Total": env.toxicities.sum(),
            "Reward": reward
        })
        state = discretize_state(next_state_cont)
        if done:
            break
    return pd.DataFrame(rows)

## 7. Smoke test (quick)

Run this first to confirm the full loop works. It uses only a few episodes and fewer integration substeps, so its metrics are **debugging outputs, not reportable final results**.

In [ ]:
smoke = train_agent("sigmoid", n_episodes=3, seed=SEED, max_days=7,
                    substeps=10, verbose=True)
print("Smoke test final tumour:", f"{smoke['tumours'][-1]:.4e}")
print("Smoke test final toxicity:", f"{smoke['toxicities'][-1]:.3f}")
assert len(smoke["rewards"]) == 3
assert np.all(np.isfinite(smoke["rewards"]))
print("Smoke test passed.")

## 8. Main experiment: compare exploration strategies

Suggested first run: 100 episodes × 3 seeds. For a stronger study, increase to 500–1000 episodes and 5–10 seeds after estimating runtime. Use the same seed list for every strategy. The environment is deterministic given an action sequence, but policy randomness differs by strategy.

Because this is tabular RL over 343 actions and a sparse discretised state space, performance can be noisy. Report means and standard deviations across seeds, not only the best run.

In [ ]:
# Edit these after the smoke test succeeds.
N_EPISODES = 100
SEEDS = [11, 22, 33]
STRATEGIES = ["epsilon_greedy", "sigmoid", "ucb"]
SUBSTEPS = 20  # use 100 for more accurate integration in final runs; runtime increases

agents = {}
records = []
start = time.time()

for strategy in STRATEGIES:
    for seed in SEEDS:
        print(f"Training {strategy} | seed={seed}")
        result = train_agent(strategy, n_episodes=N_EPISODES, seed=seed,
                             max_days=MAX_DAYS, substeps=SUBSTEPS,
                             ucb_c=1.5, verbose=False)
        agents[(strategy, seed)] = result
        for ep in range(N_EPISODES):
            records.append({
                "Strategy": strategy, "Seed": seed, "Episode": ep + 1,
                "Episode_Reward": result["rewards"][ep],
                "Final_Tumour": result["tumours"][ep],
                "Final_Toxicity": result["toxicities"][ep],
                "Epsilon": result["epsilons"][ep],
                "Exploration_Fraction": result["exploration_fraction"][ep]
            })

training_seconds = time.time() - start
results_df = pd.DataFrame(records)
print(f"Finished in {training_seconds:.1f} seconds")
display(results_df.head())

## 9. Summarise results with uncertainty across seeds

Primary outcomes:
- **Mean episode reward** in the last 20% of training (higher is better under this reward).
- **Final tumour burden** (lower is better).
- **Final total toxicity** (interpret alongside tumour burden; neither metric alone is sufficient).
- **Exploration fraction** for ε-based methods. UCB does not have an ε exploration fraction, so it is recorded as zero by this definition; do not compare that column as if UCB's exploration were zero.

Use mean ± standard deviation across seed-level averages for a more honest estimate.

In [ ]:
tail_n = max(1, int(0.20 * N_EPISODES))
seed_summaries = []
for (strategy, seed), agent in agents.items():
    seed_summaries.append({
        "Strategy": strategy,
        "Seed": seed,
        "Mean_Reward_Last20%": float(np.mean(agent["rewards"][-tail_n:])),
        "Mean_Final_Tumour_Last20%": float(np.mean(agent["tumours"][-tail_n:])),
        "Mean_Final_Toxicity_Last20%": float(np.mean(agent["toxicities"][-tail_n:])),
        "Mean_Epsilon": float(np.mean(agent["epsilons"])),
        "Mean_Exploration_Fraction": float(np.mean(agent["exploration_fraction"])),
        "Unique_States": agent["unique_states"],
        "Visited_State_Action_Pairs": agent["visited_state_action_pairs"]
    })
seed_summary_df = pd.DataFrame(seed_summaries)
display(seed_summary_df)

aggregate_df = seed_summary_df.groupby("Strategy").agg(
    Reward_Mean=("Mean_Reward_Last20%", "mean"),
    Reward_SD=("Mean_Reward_Last20%", "std"),
    Tumour_Mean=("Mean_Final_Tumour_Last20%", "mean"),
    Tumour_SD=("Mean_Final_Tumour_Last20%", "std"),
    Toxicity_Mean=("Mean_Final_Toxicity_Last20%", "mean"),
    Toxicity_SD=("Mean_Final_Toxicity_Last20%", "std")
).reset_index()
display(aggregate_df)

In [ ]:
# Learning curves: mean across seeds, shaded ±1 standard deviation
fig, ax = plt.subplots(figsize=(10, 5))
for strategy in STRATEGIES:
    subset = results_df[results_df["Strategy"] == strategy]
    grouped = subset.groupby("Episode")["Episode_Reward"]
    mean = grouped.mean()
    std = grouped.std().fillna(0)
    smooth_mean = mean.rolling(10, min_periods=1).mean()
    smooth_std = std.rolling(10, min_periods=1).mean()
    ax.plot(smooth_mean.index, smooth_mean.values, label=strategy)
    ax.fill_between(smooth_mean.index,
                    (smooth_mean - smooth_std).values,
                    (smooth_mean + smooth_std).values, alpha=0.15)
ax.set_xlabel("Episode"); ax.set_ylabel("Reward (10-episode moving average)")
ax.set_title("Double Q-learning: exploration strategy comparison")
ax.legend(); ax.grid(alpha=.25); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for strategy in STRATEGIES:
    sub = results_df[results_df["Strategy"] == strategy]
    g = sub.groupby("Episode")["Final_Tumour"]
    mean = g.mean()
    axes[0].plot(mean.index, mean.values, label=strategy)
    g2 = sub.groupby("Episode")["Final_Toxicity"]
    axes[1].plot(g2.mean().index, g2.mean().values, label=strategy)
axes[0].set_yscale("log")
axes[0].set_title("Terminal tumour burden (log scale)")
axes[0].set_xlabel("Episode"); axes[0].set_ylabel("Tumour cells")
axes[1].set_title("Terminal total toxicity")
axes[1].set_xlabel("Episode"); axes[1].set_ylabel("Toxicity index")
for ax in axes: ax.grid(alpha=.25); ax.legend()
plt.tight_layout(); plt.show()

## 10. Evaluate one trained policy per strategy

This evaluates the learned greedy policy for one selected seed, using the same evaluation horizon and no exploration bonus. These are single-seed trajectories; include multiple evaluation seeds or repeated stochastic initial conditions if you extend the simulator.

In [ ]:
EVAL_SEED = SEEDS[0]
evaluation_runs = {}
for strategy in STRATEGIES:
    agent = agents[(strategy, EVAL_SEED)]
    trajectory = evaluate_greedy_policy(agent, max_days=MAX_DAYS,
                                        seed=1000 + EVAL_SEED, substeps=SUBSTEPS)
    evaluation_runs[strategy] = trajectory
    print(strategy, "| terminal tumour:", f"{trajectory['Tumour_Cells'].iloc[-1]:.4e}",
          "| terminal toxicity:", f"{trajectory['Toxicity_Total'].iloc[-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for strategy, trajectory in evaluation_runs.items():
    axes[0].plot(trajectory["Day"], trajectory["Tumour_Cells"], label=strategy)
    axes[1].plot(trajectory["Day"], trajectory["Toxicity_Total"], label=strategy)
axes[0].set_yscale("log")
axes[0].set_title("Greedy evaluation: tumour burden")
axes[0].set_xlabel("Day"); axes[0].set_ylabel("Tumour cells")
axes[1].set_title("Greedy evaluation: total toxicity")
axes[1].set_xlabel("Day"); axes[1].set_ylabel("Toxicity index")
for ax in axes: ax.grid(alpha=.25); ax.legend()
plt.tight_layout(); plt.show()

## 11. Export outputs

The CSVs are useful for making presentation tables and reproducing plots later. The notebook itself is the implementation artifact; execute all cells in Colab before presenting any performance claims.

In [ ]:
os.makedirs("outputs", exist_ok=True)
results_df.to_csv("outputs/episode_level_results.csv", index=False)
seed_summary_df.to_csv("outputs/seed_level_summary.csv", index=False)
aggregate_df.to_csv("outputs/strategy_comparison_summary.csv", index=False)
for strategy, trajectory in evaluation_runs.items():
    trajectory.to_csv(f"outputs/evaluation_{strategy}.csv", index=False)

print("Saved:")
print("- outputs/episode_level_results.csv")
print("- outputs/seed_level_summary.csv")
print("- outputs/strategy_comparison_summary.csv")
print("- outputs/evaluation_<strategy>.csv")

## 12. How to interpret and present the study

### Proposed hypothesis
A sigmoid ε schedule may retain exploration for longer during early learning and then transition more sharply toward exploitation. UCB may systematically favour under-visited actions, potentially improving state-action coverage. Neither is guaranteed to outperform ε-greedy; the answer must come from repeated experiments.

### Minimum results to report
1. Reward learning curves averaged over seeds with variability.
2. Terminal tumour burden and terminal toxicity together.
3. Runtime and number of visited state-action pairs.
4. Sensitivity to UCB coefficient \(c\) (e.g. 0.5, 1.0, 1.5, 2.0) and sigmoid midpoint/slope.
5. Ablation: same exploration strategies with identical seeds, episode counts, discretisation, reward, and numerical integration settings.

### Limitations to disclose
- This is a reconstructed simulation, not the authors' original source code.
- The state discretisation, reward aggregation, integration method, and initial conditions are implementation assumptions.
- The action space has 343 actions, so tabular exploration is expensive and state-action coverage may be sparse.
- Results are simulation-only and cannot support patient-specific dosing or clinical conclusions.

### Suggested next steps for a stronger research contribution
- Tune exploration hyperparameters on validation seeds, then evaluate on held-out seeds.
- Add confidence intervals or bootstrap intervals across seeds.
- Test a second reward definition and alternative toxicity handling.
- Compare against the paper's ε-greedy baseline before claiming an improvement.
- Consider action-space factorisation or function approximation if tabular learning remains sample-inefficient.

**Reference:** Alizade, B. & Hajipour, A. *Double Q-Learning for Intelligent Multi-Drug Scheduling in Cancer Chemotherapy Optimisation*. IET Systems Biology, 2026, 20:e70083. DOI: 10.1049/syb2.70083.